# Ground the 3D manual back onto your photo (key-free, Colab)

Takes the **laptop-charge photo you sent** and projects the manual's actions onto it. For each step it
locates the **source object** and the **target site** in your real picture, then draws the action: an
arrow, the highlighted target, a translucent **ghost of the part at its destination**, and the friendly
sentence. Nothing is re-generated - the explanation is drawn on your actual pixels.

**Grounding model: OWLv2** (Google open-vocabulary detector, ~150M). Chosen because it is **built into
mainline transformers** - no `trust_remote_code`, no `libvips`/`pyvips`, no transformers version pin.
That sidesteps the whole Moondream dependency chain (the `_Ink`, `all_tied_weights_keys`, and
`libvips.so.42` errors). All open weights, **no API keys**.

The step **sentences are templated from the action program** (not model-generated), so we do not need a
language model at all, and the wording can never hallucinate a step.

**Colab: Runtime -> Change runtime type -> T4 GPU. Internet on by default. No restart dance needed.**

In [ ]:
# 1. Install - just mainline transformers + gradio. No apt, no pyvips, no pins.
!pip install -q -U transformers accelerate gradio
import transformers; print("transformers", transformers.__version__)

In [ ]:
# 2. Load OWLv2 (no key, no remote code)
import math, textwrap
import torch
from transformers import Owlv2Processor, Owlv2ForObjectDetection
from PIL import Image, ImageDraw, ImageFont

DEV = "cuda" if torch.cuda.is_available() else "cpu"
proc = Owlv2Processor.from_pretrained("google/owlv2-base-patch16-ensemble")
det  = Owlv2ForObjectDetection.from_pretrained("google/owlv2-base-patch16-ensemble").to(DEV).eval()

try:
    FONT  = ImageFont.truetype("DejaVuSans.ttf", 20)
    FONT_S = ImageFont.truetype("DejaVuSans.ttf", 16)
except Exception:
    FONT = FONT_S = ImageFont.load_default()
print("OWLv2 loaded on", DEV)

In [ ]:
# 3. The manual program + grounding/overlay logic

# Action program (matches laptop-charge-002). find_* are detector phrases (OWLv2 likes a few synonyms);
# `say` is the grounded friendly sentence (templated from the action, so it can't hallucinate steps).
STEPS = [
    dict(n=1, find_src=["white plug head", "small power plug adapter", "wall plug prong"],
              find_tgt=["white laptop power adapter", "white charger brick"],
         say="Clip the flat-blade (North American) plug head onto the back of the white adapter until it seats. Leave the other head aside."),
    dict(n=2, find_src=["white laptop power adapter", "white charger brick"],
              find_tgt=["power strip", "black power strip with outlets"],
         say="Plug the adapter into an empty outlet on the black power strip."),
    dict(n=3, find_src=["cable connector tip", "magsafe connector", "end of charging cable"],
              find_tgt=["laptop", "open laptop computer"],
         say="Connect the MagSafe connector to the laptop's charging port - it holds magnetically and the light comes on."),
]

def resize_long(img, m=1024):
    W, H = img.size; s = m/max(W, H)
    return img.resize((int(W*s), int(H*s))) if s < 1 else img

def ground_one(image, phrases):
    # OWLv2 open-vocabulary detection; try several phrasings, keep the highest-scoring box
    inp = proc(text=[phrases], images=image, return_tensors="pt").to(DEV)
    with torch.no_grad():
        out = det(**inp)
    tsz = torch.tensor([[image.size[1], image.size[0]]]).to(DEV)  # (H, W)
    try:
        res = proc.post_process_object_detection(outputs=out, threshold=0.02, target_sizes=tsz)[0]
    except Exception:
        res = proc.post_process_grounded_object_detection(outputs=out, threshold=0.02, target_sizes=tsz)[0]
    boxes, scores = res["boxes"], res["scores"]
    if len(boxes) == 0:
        return None
    i = int(scores.argmax())
    return tuple(float(v) for v in boxes[i])

def center(b):
    return ((b[0]+b[2])/2.0, (b[1]+b[3])/2.0)

def arrow(d, p0, p1, color, w):
    d.line([p0, p1], fill=color, width=w)
    a = math.atan2(p1[1]-p0[1], p1[0]-p0[0]); L = 20
    for da in (2.6, -2.6):
        d.line([p1, (p1[0]+L*math.cos(a+da), p1[1]+L*math.sin(a+da))], fill=color, width=w)

def caption(img, text, ok):
    d = ImageDraw.Draw(img); W, H = img.size
    lines = textwrap.wrap(text, width=max(20, int(W/12)))
    if not ok:
        lines += ["[some objects not located - try a clearer photo or edit find_src/find_tgt]"]
    lh = 26; bar = 10 + lh*len(lines)
    d.rectangle([0, H-bar, W, H], fill=(0, 0, 0))
    for i, ln in enumerate(lines):
        col = (255, 210, 0) if (not ok and i == len(lines)-1) else (255, 255, 255)
        d.text((10, H-bar+6+i*lh), ln, fill=col, font=FONT)

def overlay_step(image, step):
    base = image.convert("RGBA"); W, H = base.size
    src = ground_one(image, step["find_src"])
    tgt = ground_one(image, step["find_tgt"])
    layer = Image.new("RGBA", base.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(layer); w = max(3, W//250)
    if src and tgt:                                   # ghost the source at the target
        sb = tuple(int(v) for v in src)
        crop = base.crop(sb).copy(); crop.putalpha(120)
        tcx, tcy = center(tgt); cw, ch = crop.size
        layer.alpha_composite(crop, (int(tcx-cw/2), int(tcy-ch/2)))
        arrow(d, center(src), (tcx, tcy), (255, 140, 0, 255), w)
    if src:
        d.rectangle(src, outline=(255, 140, 0, 255), width=w)
        d.text((src[0], max(0, src[1]-18)), "take", fill=(255, 140, 0, 255), font=FONT_S)
    if tgt:
        d.rectangle(tgt, outline=(220, 40, 40, 255), width=w)
        d.text((tgt[0], max(0, tgt[1]-18)), "here", fill=(220, 40, 40, 255), font=FONT_S)
    out = Image.alpha_composite(base, layer).convert("RGB")
    caption(out, "Step %d: %s" % (step["n"], step["say"]), bool(src and tgt))
    return out

print("grounding logic ready")

In [ ]:
# 4. Interface: upload your photo -> grounded manual
import gradio as gr

def run(image):
    if image is None:
        return [], "Upload the laptop-charge photo first."
    image = resize_long(image.convert("RGB"), 1024)
    gallery = [(overlay_step(image, s), "Step %d" % s["n"]) for s in STEPS]
    notes = "\n".join("%d. %s" % (s["n"], s["say"]) for s in STEPS)
    return gallery, notes

with gr.Blocks(title="Ground the manual on your photo") as demo:
    gr.Markdown("## Upload your scene photo -> the manual is drawn onto it")
    in_img = gr.Image(type="pil", label="Your photo (the laptop-charge scene)")
    go = gr.Button("Ground the manual", variant="primary")
    out_gallery = gr.Gallery(label="Grounded steps", columns=3, height=420)
    out_notes = gr.Markdown()
    go.click(run, [in_img], [out_gallery, out_notes])

demo.launch(share=True, debug=False)

## What you are seeing
- **Orange = take this part**, **red = put it here**, arrow = the move, faint copy = a **ghost of the
  part at its destination**. The black bar is the grounded sentence.
- OWLv2 is open-vocabulary but small, so some boxes will be off (especially the tiny MagSafe tip and an
  "empty outlet"). Tune the `find_src` / `find_tgt` phrase lists in cell 3. Best-effort grounding like
  this is exactly the perception gap your research targets - not a UX bug.

## Why this model
OWLv2 is a first-class transformers model: no `trust_remote_code`, no native libs, no version pin. If
you ever want language too (auto-generated descriptions instead of templated ones), add a VLM later -
but for grounding-to-pixels, a detector is the robust, dependency-light choice.

## Stronger grounding (still key-free)
- **SAM2** on each OWLv2 box -> highlight/ghost follows the real silhouette, not a rectangle.
- **Depth Anything v2** on the masks -> a coarse surface normal per part = a real 3D-ish pose.
- **FoundationPose / MegaPose** (needs a CAD/3D proxy) -> full 6-DoF, the path back to the
  IKEA-Manuals-at-Work poses and your 3D state machine.

The program (STEPS) and renderer stay the same; you only swap the perception that turns the photo into
part locations.